In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [9]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

python_for_beginners = skyline[skyline["course_name"] == "Python for Beginners"]["hours_studied"]
sql_basics = skyline[skyline["course_name"] == "SQL Basics"]["hours_studied"]

print(f"Python for Beginners: n={len(python_for_beginners)}, mean={python_for_beginners.mean():.0f}, std={python_for_beginners.std():.0f}")
print(f"SQL Basics: n={len(sql_basics)}, mean={sql_basics.mean():.0f}, std={sql_basics.std():.0f}")


Python for Beginners: n=24, mean=31, std=5
SQL Basics: n=24, mean=29, std=5


In [10]:
t_stat, p_value = stats.ttest_ind(python_for_beginners, sql_basics)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 0.9618
p-value: 0.3412


In [18]:
mean_diff = python_for_beginners.mean() - sql_basics.mean()

# Pooled standard error of the difference
n1, n2 = len(python_for_beginners), len(sql_basics)
s1, s2 = python_for_beginners.std(ddof=1), sql_basics.std(ddof=1)
se_diff = np.sqrt(s1**2 / n1 + s2**2 / n2)

# 95% CI for the difference
margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

print(f"Mean difference (Python for Beginners - SQL Basics): {mean_diff:.2f}")
print(f"95% CI for the difference: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"p-value: {p_value:.4f}")

Mean difference (Python for Beginners - SQL Basics): 1.38
95% CI for the difference: (-1.43, 4.18)
p-value: 0.3412


While students in python for beginners studied about 1.4 hours more per week than those in SQL basics, this difference is not statistically meaningul, we cannot reliably conclude that one course actually requires more study time, since the p-value of 0.34 is well above the standard threshold of 0.05. Even if a real difference exists, the 95% CI shows the true difference is likely small, suggesting both courses demand similar study efforts from students.

In [31]:
np.random.seed(2024)

n_tests = 10
significant_count = 0
all_p_values = []

for i in range(n_tests):
    # Two samples from the SAME distribution (null is true)
    course_a = np.random.normal(loc=100, scale=15, size=80)
    course_b = np.random.normal(loc=100, scale=15, size=80)

    _, p = stats.ttest_ind(course_a, course_b)
    all_p_values.append(p)
    
    if p < 0.05:
        significant_count += 1
        print(f"Test {i+1}: p = {p:.4f} (SIGNIFICANT, but null is actually true)")

print(f"\nOf {n_tests} tests on identical distributions, {significant_count} came back 'significant' at p < 0.05")


Test 7: p = 0.0193 (SIGNIFICANT, but null is actually true)

Of 10 tests on identical distributions, 1 came back 'significant' at p < 0.05


In [22]:
bonferroni_threshold = 0.05 / n_tests
significant_after_bonferroni = sum(p < bonferroni_threshold for p in all_p_values)

print(f"Original threshold: 0.05")
print(f"Bonferroni-corrected threshold: {bonferroni_threshold:.4f}")
print(f"Number significant at original threshold: {significant_count}")
print(f"Number significant after Bonferroni: {significant_after_bonferroni}")

Original threshold: 0.05
Bonferroni-corrected threshold: 0.0050
Number significant at original threshold: 1
Number significant after Bonferroni: 0


In [34]:
import itertools

courses = skyline["course_name"].unique()
course_pairs = list(itertools.combinations(courses, 2))

results = []

for course_a, course_b in course_pairs:
    data_a = skyline[skyline["course_name"] == course_a]["hours_studied"]
    data_b = skyline[skyline["course_name"] == course_b]["hours_studied"]
    
    t_stat, p_value = stats.ttest_ind(data_a, data_b)
    mean_diff = data_a.mean() - data_b.mean()
    
    results.append({
        "course_a": course_a,
        "course_b": course_b,
        "mean_difference": mean_diff,
        "p_value": p_value
    })

results_df = pd.DataFrame(results)
results_df.to_csv("pairwise_ttest_results.csv", index=False)
print(results_df)

               course_a              course_b  mean_difference   p_value
0  Tableau Fundamentals  Python for Beginners        -0.630000  0.711274
1  Tableau Fundamentals            SQL Basics         0.745000  0.657690
2  Tableau Fundamentals        Statistics 101        -0.803077  0.715884
3  Tableau Fundamentals    Intro to Analytics        -1.308571  0.515376
4  Python for Beginners            SQL Basics         1.375000  0.341169
5  Python for Beginners        Statistics 101        -0.173077  0.925529
6  Python for Beginners    Intro to Analytics        -0.678571  0.678620
7            SQL Basics        Statistics 101        -1.548077  0.395694
8            SQL Basics    Intro to Analytics        -2.053571  0.203176
9        Statistics 101    Intro to Analytics        -0.505495  0.801705


0 of the 10 pairwise comparisons showed p<0.05 at the standard significance level. which indicates no detectable differences in hours studied between courses. 

The Bonferroni-corrected threshold for 10 tests at the 0.05 family-wise error rate is 0.005. This is much stricter than the standard 0.05 threshold. Instead of checking each test individually at p < 0.05, Bonferroni divides 0.05 by 10 to account for the fact that running many tests increases your chances of finding false positives. This correction ensures that across all 10 tests combined, you have only a 5% chance of making any false positive.

Since none of your tests reached p < 0.05, zero tests survive the Bonferroni correction.

The simulation with identical distributions shows why the multiple testing trap is dangerous. When you run many tests on data with no real differences, some will appear "significant" just by random chance.

